In [0]:
sas_token = dbutils.secrets.get(
    scope="uber-eventhub",
    key="adls-sas-token"
)

In [0]:
import pandas as pd

files = [
{"file": "map_cities"},
{"file":"map_cancellation_reasons"},
{"file":"map_payment_methods"},
{"file":"map_ride_statuses"},
{"file":"map_vehicle_makes"},
{"file":"map_vehicle_types"}
]

for file in files:

    url = f"https://nooruberstorage.blob.core.windows.net/raw/ingestion/{file['file']}.json?{sas_token}"

    df = pd.read_json(url)
    df_spark = spark.createDataFrame(df)
    
    # Writing Data To the Bronze Layer
    df_spark.write.format("delta")\
            .mode("overwrite")\
            .option("overwriteSchema", "true")\
            .saveAsTable(f"uber.bronze.{file['file']}")

In [0]:
url = f"https://nooruberstorage.blob.core.windows.net/raw/ingestion/bulk_rides.json?{sas_token}"

df = pd.read_json(url)
df_spark = spark.createDataFrame(df)
if not spark.catalog.tableExists("uber.bronze.bulk_rides"):
    df_spark.write.format("delta")\
            .mode("overwrite")\
            .saveAsTable(f"uber.bronze.bulk_rides")
    print("This will not run more than 1 time")